In [1]:
!pip install sentence-transformers faiss-cpu scikit-fuzzy scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 68.6 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import re

from sklearn.datasets import fetch_20newsgroups
from sentence_transformers import SentenceTransformer
import faiss

import skfuzzy as fuzz
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
from google.colab import files
uploaded = files.upload()

Saving 20_newsgroups.tar.gz to 20_newsgroups.tar.gz


In [4]:
import tarfile

tar = tarfile.open("20_newsgroups.tar.gz")

tar.extractall()

tar.close()

print("Dataset extracted successfully")

/tmp/ipykernel_495/2008702916.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall()


Dataset extracted successfully


In [5]:
import os

dataset_path = "20_newsgroups"

categories = os.listdir(dataset_path)

print("Total categories:", len(categories))
print(categories)

Total categories: 20
['comp.sys.mac.hardware', 'sci.med', 'comp.os.ms-windows.misc', 'rec.sport.baseball', 'rec.autos', 'rec.sport.hockey', 'sci.space', 'talk.religion.misc', 'soc.religion.christian', 'comp.sys.ibm.pc.hardware', 'comp.windows.x', 'sci.crypt', 'talk.politics.mideast', 'talk.politics.misc', 'rec.motorcycles', 'misc.forsale', 'comp.graphics', 'sci.electronics', 'alt.atheism', 'talk.politics.guns']


In [6]:
documents = []
labels = []
label_names = []

for category in categories:

    category_path = os.path.join(dataset_path, category)

    label_names.append(category)

    for file in os.listdir(category_path):

        file_path = os.path.join(category_path, file)

        try:
            with open(file_path, "r", encoding="latin1") as f:
                text = f.read()

                documents.append(text)
                labels.append(category)

        except:
            pass

print("Total Documents:", len(documents))

Total Documents: 19997


In [7]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text


clean_documents = [clean_text(doc) for doc in documents]

print(clean_documents[0][:500])

newsgroups compsysmachardware path cantaloupesrvcscmuedumagnesiumclubcccmuedunewsseicmueducisohiostateeduzaphodmpsohiostateeduuwmeduuxcsouiucedunewsrelayiastateedunewsiastateedukmradke from kmradkeiastateedu kevin m radke subject unknown mac board national instruments nbdma messageid cygdqminewsiastateedu sender newsnewsiastateedu usenet news system organization iowa state university ames ia distribution usa date fri apr gmt lines i need help identifying this board that i found stuffed away in a


In [8]:
import re

def clean_text(text):

    text = text.lower()

    text = re.sub(r'\n', ' ', text)

    text = re.sub(r'[^a-zA-Z ]', '', text)

    text = re.sub(r'\s+', ' ', text)

    return text


clean_documents = [clean_text(doc) for doc in documents]

print(clean_documents[0][:500])

newsgroups compsysmachardware path cantaloupesrvcscmuedumagnesiumclubcccmuedunewsseicmueducisohiostateeduzaphodmpsohiostateeduuwmeduuxcsouiucedunewsrelayiastateedunewsiastateedukmradke from kmradkeiastateedu kevin m radke subject unknown mac board national instruments nbdma messageid cygdqminewsiastateedu sender newsnewsiastateedu usenet news system organization iowa state university ames ia distribution usa date fri apr gmt lines i need help identifying this board that i found stuffed away in a


In [9]:
!pip install sentence-transformers

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    clean_documents,
    batch_size=32,
    show_progress_bar=True
)

embeddings = np.array(embeddings)

print("Embedding Shape:", embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Embedding Shape: (19997, 384)


In [11]:
!pip install faiss-cpu

In [12]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Total vectors:", index.ntotal)

Total vectors: 19997


In [13]:
def semantic_search(query, top_k=5):

    query_embedding = model.encode([query])

    distances, indices = index.search(query_embedding, top_k)

    results = []

    for idx in indices[0]:
        results.append(documents[idx][:400])

    return results

In [14]:
results = semantic_search("space shuttle launch")

for r in results:
    print(r)
    print("------------")

Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!pitt.edu!gatech!emory!news-feed-1.peachnet.edu!darwin.sura.net!howland.reston.ans.net!noc.near.net!uunet!pipex!uknet!bradford.ac.uk!C.H.A.Wong
From: C.H.A.Wong@bradford.ac.uk (CHA WONG)
Newsgroups: sci.space
Subject: How can you see the launch of the Space Shuttle ?
Message-ID: <1993Apr25.031940.9794@bradford.ac.uk>
Date: 25 Apr 93 03:19:40
------------
Xref: cantaloupe.srv.cs.cmu.edu sci.space:59904 sci.answers:108 news.answers:7223
Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv.cs.cmu.edu!fs7.ece.cmu.edu!europa.eng.gtefsd.com!gatech!concert!borg.cs.unc.edu!not-for-mail
From: leech@cs.unc.edu (Jon Leech)
Newsgroups: sci.space,sci.answers,news.answers
Subject: Space FAQ 09/15 - Mission Schedules
Supersedes: <schedule_730956538@cs.unc.edu>
Followup
------------
Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!noc.near.net!uunet!digex.com!digex.net!not-for-mail
From: prb@access.digex.net (Pat)
Newsgroups: sci.space
Subject: Re:

In [15]:
!pip install scikit-fuzzy

In [16]:
import skfuzzy as fuzz

data = embeddings.T

clusters = 10

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    data,
    c=clusters,
    m=2,
    error=0.005,
    maxiter=1000
)

print("Cluster centers shape:", cntr.shape)

Cluster centers shape: (10, 384)


In [17]:
membership = u.T

print("Membership shape:", membership.shape)

print("Example cluster distribution:")
print(membership[0])

Membership shape: (19997, 10)
Example cluster distribution:
[0.09999779 0.09999905 0.10000167 0.10000146 0.09999822 0.10000257
 0.09999991 0.09999762 0.10000103 0.10000067]


In [18]:
np.save("embeddings.npy", embeddings)

np.save("cluster_memberships.npy", membership)

faiss.write_index(index, "vector_index.faiss")

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class SemanticCache:

    def __init__(self, threshold=0.85):

        self.cache = []
        self.threshold = threshold

        self.hit_count = 0
        self.miss_count = 0


    def lookup(self, query_embedding):

        if len(self.cache) == 0:
            self.miss_count += 1
            return False, None, None, None

        cached_embeddings = np.array([entry["embedding"] for entry in self.cache])

        similarities = cosine_similarity(
            [query_embedding],
            cached_embeddings
        )[0]

        best_idx = np.argmax(similarities)
        best_score = similarities[best_idx]

        if best_score >= self.threshold:

            self.hit_count += 1

            entry = self.cache[best_idx]

            return True, entry["query"], entry["result"], best_score

        else:

            self.miss_count += 1

            return False, None, None, best_score


    def store(self, query, embedding, result, cluster):

        self.cache.append({
            "query": query,
            "embedding": embedding,
            "result": result,
            "cluster": cluster
        })


    def stats(self):

        total = len(self.cache)

        hit_rate = self.hit_count / (self.hit_count + self.miss_count + 1e-9)

        return {
            "total_entries": total,
            "hit_count": self.hit_count,
            "miss_count": self.miss_count,
            "hit_rate": round(hit_rate, 3)
        }


    def clear(self):

        self.cache = []

        self.hit_count = 0
        self.miss_count = 0

In [21]:
cache = SemanticCache(threshold=0.85)

In [25]:
def query_system(query):

    query_embedding = model.encode(query)

    hit, matched_query, result, score = cache.lookup(query_embedding)

    if hit:

        print("CACHE HIT")

        return {
            "query": query,
            "cache_hit": True,
            "matched_query": matched_query,
            "similarity_score": float(score),
            "result": result
        }

    else:

        print("CACHE MISS")

        distances, indices = index.search(
            np.array([query_embedding]),
            5
        )

        best_doc = documents[indices[0][0]]

        cluster_id = int(np.argmax(membership[indices[0][0]]))

        cache.store(
            query,
            query_embedding,
            best_doc,
            cluster_id
        )

        return {
            "query": query,
            "cache_hit": False,
            "matched_query": None,
            "similarity_score": float(score),
            "result": best_doc,
            "dominant_cluster": cluster_id
        }

In [26]:
query_system("space shuttle launch")

CACHE HIT


{'query': 'space shuttle launch',
 'cache_hit': True,
 'matched_query': 'space shuttle launch',
 'similarity_score': 0.9999999403953552,
 'result': "Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!pitt.edu!gatech!emory!news-feed-1.peachnet.edu!darwin.sura.net!howland.reston.ans.net!noc.near.net!uunet!pipex!uknet!bradford.ac.uk!C.H.A.Wong\nFrom: C.H.A.Wong@bradford.ac.uk (CHA WONG)\nNewsgroups: sci.space\nSubject: How can you see the launch of the Space Shuttle ?\nMessage-ID: <1993Apr25.031940.9794@bradford.ac.uk>\nDate: 25 Apr 93 03:19:40 GMT\nOrganization: University of Bradford, UK\nLines: 28\nX-Newsreader: TIN [version 1.1 PL9]\n\n\nSorry for asking a question that's not entirely based on the\ntechnical aspects of space, but I couldn't find the\nanswer on the FAQs !\n\nI'm currently in the UK, which makes seeing a Space Shuttle\nlaunch a little difficult.....\n\nHowever, I have been selected to be an exchange student\nat Louisiana State Uni. from August, and I am absolutel

In [28]:
query_system("nasa rocket launch")

CACHE HIT


{'query': 'nasa rocket launch',
 'cache_hit': True,
 'matched_query': 'nasa rocket launch',
 'similarity_score': 1.0000001192092896,
 'result': "Path: cantaloupe.srv.cs.cmu.edu!das-news.harvard.edu!noc.near.net!uunet!digex.com!digex.net!not-for-mail\nFrom: prb@access.digex.net (Pat)\nNewsgroups: sci.space\nSubject: Re: Political banner in space\nDate: 1 May 1993 15:02:37 -0400\nOrganization: Express Access Online Communications USA\nLines: 13\nMessage-ID: <1ruhgd$cvh@access.digex.net>\nReferences: <1rjrv2$eh3@network.ucsd.edu> <1rmfrnINN8hh@gap.caltech.edu>\nNNTP-Posting-Host: access.digex.net\n\n\nWell,  you better not get the shuttle as your launch vehicle.\n\nand most ELV's have too  far of a backlog for political messages.\n\nIf during the campaign season,  the candidates for president had\nlaunched one,  right around now we'd  be getting a launch\nfor PEROT 92.\n\nand if they had used the shuttle,  we'd be seeing launches\nfor NIXON now more then ever.\n\npat\n"}

In [29]:
cache.stats()

{'total_entries': 2, 'hit_count': 3, 'miss_count': 2, 'hit_rate': 0.6}

In [30]:
cache.clear()

cache.stats()

{'total_entries': 0, 'hit_count': 0, 'miss_count': 0, 'hit_rate': 0.0}